# Balansis API Examples

This notebook is a compact API-first walkthrough of the current Balansis runtime surface.

## Coverage

1. `AbsoluteValue`
2. low-level `Operations`
3. `EternalRatio`
4. `Compensator`
5. runtime algebra helpers
6. NumPy bridge helpers

In [ ]:
import numpy as np

from balansis import (
    AbsoluteValue,
    EternalRatio,
    Operations,
    Compensator,
    AbsoluteGroup,
    EternityField,
    ABSOLUTE,
)
from balansis.algebra.absolute_group import GroupElement
from balansis.algebra.eternity_field import FieldElement
from balansis.numpy_integration import compensated_array_add, compensated_softmax

print('Balansis API walkthrough imports loaded.')

## 1. `AbsoluteValue` Basics

In [ ]:
positive = AbsoluteValue.from_float(5.0)
negative = AbsoluteValue.from_float(-3.0)

print('positive =', positive)
print('negative =', negative)
print('ABSOLUTE =', ABSOLUTE)
print('positive.is_positive() =', positive.is_positive())
print('negative.is_negative() =', negative.is_negative())
print('ABSOLUTE.is_absolute() =', ABSOLUTE.is_absolute())

## 2. Low-Level `Operations`

In [ ]:
add_result, add_comp = Operations.compensated_add(
    AbsoluteValue.from_float(1e16),
    AbsoluteValue.from_float(-1e16),
)
mul_result, mul_comp = Operations.compensated_multiply(
    AbsoluteValue.from_float(4.0),
    AbsoluteValue.from_float(0.5),
)
seq_result, seq_comp = Operations.sequence_sum([
    AbsoluteValue.from_float(1e16),
    AbsoluteValue.from_float(1.0),
    AbsoluteValue.from_float(-1e16),
])

print('compensated_add ->', add_result, add_comp)
print('compensated_multiply ->', mul_result, mul_comp)
print('sequence_sum ->', seq_result, seq_comp)

## 3. `EternalRatio`

In [ ]:
ratio = EternalRatio(
    numerator=AbsoluteValue.from_float(6.0),
    denominator=AbsoluteValue.from_float(2.0),
)

print('ratio =', ratio)
print('ratio.value() =', ratio.value())
print('ratio.numerical_value() =', ratio.numerical_value())
print('ratio.signed_value() =', ratio.signed_value())
print('ratio.is_stable() =', ratio.is_stable())

try:
    EternalRatio(numerator=AbsoluteValue.from_float(1.0), denominator=ABSOLUTE)
except ValueError as exc:
    print('denominator guard:', exc)

## 4. `Compensator`

In [ ]:
compensator = Compensator()
values = [
    AbsoluteValue.from_float(1e-10),
    AbsoluteValue.from_float(-1e10),
    AbsoluteValue.from_float(1.0),
    AbsoluteValue.from_float(-1e-15),
]

print('stability score =', compensator.analyze_stability(values))
print('compensate_addition =', compensator.compensate_addition(
    AbsoluteValue.from_float(10.0),
    AbsoluteValue.from_float(-9.999999999999),
))

## 5. Runtime Algebra Helpers

In [ ]:
group = AbsoluteGroup.additive_group()
g1 = GroupElement(value=AbsoluteValue.from_float(2.0))
g2 = GroupElement(value=AbsoluteValue.from_float(-3.0))

field = EternityField.rational_field()
f1 = FieldElement(ratio=EternalRatio(
    numerator=AbsoluteValue.from_float(3.0),
    denominator=AbsoluteValue.from_float(2.0),
))
f2 = FieldElement(ratio=EternalRatio(
    numerator=AbsoluteValue.from_float(4.0),
    denominator=AbsoluteValue.from_float(3.0),
))

print('group.operate(g1, g2) =', group.operate(g1, g2))
print('field.add(f1, f2) =', field.add(f1, f2))
print('field.multiply(f1, f2) =', field.multiply(f1, f2))

## 6. NumPy Bridge Helpers

In [ ]:
left = np.array([1e16, 1.0, -1e16], dtype=np.float64)
right = np.array([0.25, -0.25, 1.0], dtype=np.float64)
logits = np.array([1000.0, 999.0, 998.0], dtype=np.float64)

print('compensated_array_add(left, right) =', compensated_array_add(left, right))
print('compensated_softmax(logits) =', compensated_softmax(logits))

## Further Reading

- `docs/api/index.md`
- `docs/examples/api-examples.md`
- `docs/guides/integration-patterns.md`
- `docs/formal/proof-map.md`